# From Machine Learning Basics to Neural Networks & Deep Learning

In previous sessions, we covered **Linear Regression**, **Classification**, and **Gradient Descent**. Today, we will see how those exact building blocks combine to form **Neural Networks** and **Deep Learning**.

### Roadmap
1. **Biological vs. Artificial Neuron** — The intuition and math behind a single neuron
2. **The Perceptron** — The simplest artificial neuron (step function)
3. **The Sigmoid Neuron** — Making neurons smooth so Gradient Descent can train them
4. **Multi-Layer Networks (2–3 Layers)** — Stacking neurons to enter Deep Learning
5. **Classification Network** — Solving non-linear problems with hidden layers
6. **Softmax** — Extending neural networks to multi-class classification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from sklearn.datasets import make_moons, make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

---
# 1. The Neuron: Biological Inspiration vs. Artificial Neuron

An artificial neuron is loosely inspired by how neurons in the human brain process signals:

| Biological Neuron | Artificial Neuron | Role |
| :--- | :--- | :--- |
| **Dendrites** | Inputs ($x_1, x_2, \dots, x_n$) | Receive incoming signals (features) |
| **Synapses** | Weights ($w_1, w_2, \dots, w_n$) | Control the strength/importance of each input |
| **Cell Body (Soma)** | Weighted Sum + Bias ($z$) | Combines all inputs and adds a threshold offset ($b$) |
| **Firing Threshold** | Activation Function ($f(z)$) | Decides whether/how strongly the neuron activates |
| **Axon** | Output ($y$) | Passes the final signal to the next neuron or output |

### The Math (Notice how familiar this is!)
1. **Linear combination** (just like Linear Regression!):
$$z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \mathbf{w}^T \mathbf{x} + b$$

2. **Activation function** (adds non-linearity / decision behavior):
$$y = f(z)$$

In [ ]:
# Visual diagram of an Artificial Neuron mapped to Biological terms
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

# Input nodes (Dendrites)
inputs = [(1.2, 4.8, r'$x_1$'), (1.2, 3.0, r'$x_2$'), (1.2, 1.2, r'$x_3$')]
weights = [r'$w_1$', r'$w_2$', r'$w_3$']

for (x, y, label), w in zip(inputs, weights):
    circle = Circle((x, y), 0.4, facecolor='#dceefb', edgecolor='#1f77b4', lw=2)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=13, fontweight='bold')
    
    # Arrows (Synapses)
    arrow = FancyArrowPatch((x + 0.42, y), (4.15, 3.0), arrowstyle='->', mutation_scale=18, lw=1.8, color='#555')
    ax.add_patch(arrow)
    ax.text((x + 4.15)/2 - 0.1, (y + 3.0)/2 + 0.25, w, fontsize=12, color='#d62728', fontweight='bold')

# Soma (Weighted Sum)
soma = Circle((4.8, 3.0), 0.65, facecolor='#fff3cd', edgecolor='#ff9800', lw=2)
ax.add_patch(soma)
ax.text(4.8, 3.0, r'$\Sigma + b$', ha='center', va='center', fontsize=13, fontweight='bold')

# Bias arrow
ax.annotate('', xy=(4.8, 3.68), xytext=(4.8, 4.9), arrowprops=dict(arrowstyle='->', lw=1.8, color='#555'))
ax.text(4.8, 5.1, 'Bias (b)', ha='center', fontsize=11, fontweight='bold', color='#555')

# Activation block
ax.annotate('', xy=(6.45, 3.0), xytext=(5.48, 3.0), arrowprops=dict(arrowstyle='->', lw=1.8, color='#555'))
ax.text(5.95, 3.25, 'z', ha='center', fontsize=12, fontweight='bold')

act = Circle((7.0, 3.0), 0.55, facecolor='#e8f5e9', edgecolor='#2e7d32', lw=2)
ax.add_patch(act)
ax.text(7.0, 3.0, r'$f(z)$', ha='center', va='center', fontsize=13, fontweight='bold')

# Output (Axon)
ax.annotate('', xy=(8.9, 3.0), xytext=(7.58, 3.0), arrowprops=dict(arrowstyle='->', lw=2, color='#2e7d32'))
ax.text(9.2, 3.0, r'$y$', ha='center', va='center', fontsize=15, fontweight='bold', color='#2e7d32')

# Biological Labels
ax.text(1.2, 0.3, 'Inputs\n(Dendrites)', ha='center', fontsize=10, color='#1f77b4', fontweight='bold')
ax.text(2.8, 0.3, 'Weights\n(Synapses)', ha='center', fontsize=10, color='#d62728', fontweight='bold')
ax.text(4.8, 0.3, 'Weighted Sum\n(Cell Body / Soma)', ha='center', fontsize=10, color='#e65100', fontweight='bold')
ax.text(7.0, 0.3, 'Activation\n(Firing Threshold)', ha='center', fontsize=10, color='#2e7d32', fontweight='bold')
ax.text(9.0, 0.3, 'Output\n(Axon)', ha='center', fontsize=10, color='#2e7d32', fontweight='bold')

plt.title('Structure of an Artificial Neuron (with Biological Analogy)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 2. A Simple Neuron: The Perceptron

The **Perceptron** (1958) is the earliest and simplest artificial neuron. It uses a **Step Function** as its activation:

$$f(z) = \begin{cases} 1 & \text{if } z \ge 0 \\ 0 & \text{if } z < 0 \end{cases}$$

If the combined signal $z = w_1 x_1 + w_2 x_2 + b$ crosses $0$, the neuron **fires (1)**; otherwise it stays **off (0)**.

In [ ]:
# Define and plot the Step Function
def step_function(z):
    return np.where(z >= 0, 1, 0)

z = np.linspace(-6, 6, 400)
plt.figure(figsize=(7, 3.5))
plt.plot(z, step_function(z), color='#1f77b4', lw=2.5, label='Step Function')
plt.axvline(0, color='gray', linestyle='--', alpha=0.6)
plt.title('Perceptron Activation: Step Function')
plt.xlabel('z (Weighted Sum + Bias)')
plt.ylabel('Output f(z)')
plt.ylim(-0.2, 1.2)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### Coding a Perceptron from Scratch
Let's build a Perceptron to model a simple **AND gate**: it should only output `1` when both $x_1 = 1$ and $x_2 = 1$.

In [ ]:
def perceptron(x, weights, bias):
    z = np.dot(x, weights) + bias
    return step_function(z)

# AND gate inputs and labels
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

# Hand-picked weights and bias for AND
w = np.array([1.0, 1.0])
b = -1.5

preds = perceptron(X_and, w, b)
for xi, yi, pi in zip(X_and, y_and, preds):
    print(f"Input: {xi} | True: {yi} | Perceptron Output: {pi}")

In [ ]:
# Plot the Perceptron's linear decision boundary: w1*x1 + w2*x2 + b = 0
x1_vals = np.linspace(-0.2, 1.4, 100)
x2_vals = -(w[0] * x1_vals + b) / w[1]

plt.figure(figsize=(5.5, 4.5))
plt.scatter(X_and[y_and == 0, 0], X_and[y_and == 0, 1], c='blue', s=120, edgecolors='k', label='Class 0')
plt.scatter(X_and[y_and == 1, 0], X_and[y_and == 1, 1], c='red', s=120, edgecolors='k', label='Class 1')
plt.plot(x1_vals, x2_vals, 'k--', lw=2, label='Decision Boundary (z = 0)')
plt.xlim(-0.2, 1.4)
plt.ylim(-0.2, 1.4)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Perceptron Solving the AND Gate')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
# 3. The Sigmoid Neuron

### Why isn't the Perceptron enough?
Remember **Gradient Descent**? To learn weights automatically, we need the **derivative (slope)** of the output with respect to the weights.
- The Step Function is completely **flat** everywhere (slope = 0) and jumps abruptly at $z = 0$.
- A tiny change in weights can cause the output to flip from 0 to 1—or do nothing at all. Gradient Descent cannot work with that!

### Solution: The Sigmoid Function
We replace the hard step with a smooth S-shaped curve (the exact same function used in **Logistic Regression**!):

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

And its derivative is clean and smooth:
$$\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$$

Now, a small change in weights produces a **small, predictable change in output** (a probability between 0 and 1).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

z = np.linspace(-8, 8, 400)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Step vs Sigmoid
axes[0].plot(z, step_function(z), 'k--', lw=1.8, alpha=0.6, label='Step (Perceptron)')
axes[0].plot(z, sigmoid(z), color='#e65100', lw=2.5, label='Sigmoid $\\sigma(z)$')
axes[0].set_title('Step Function vs. Sigmoid Function')
axes[0].set_xlabel('z')
axes[0].set_ylabel('Output')
axes[0].grid(alpha=0.3)
axes[0].legend()

# Right: Sigmoid Derivative
axes[1].plot(z, sigmoid_derivative(z), color='#2e7d32', lw=2.5, label="Sigmoid Derivative $\\sigma'(z)$")
axes[1].set_title('Smooth Gradient for Gradient Descent')
axes[1].set_xlabel('z')
axes[1].set_ylabel('Slope')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Sigmoid Neuron in action: outputs smooth probabilities instead of hard 0/1
def sigmoid_neuron(x, weights, bias):
    z = np.dot(x, weights) + bias
    return sigmoid(z)

probs = sigmoid_neuron(X_and, weights=np.array([5.0, 5.0]), bias=-7.5)
for xi, p in zip(X_and, probs):
    print(f"Input: {xi} -> Probability of Class 1: {p:.4f}")

---
# 4. Stepping into Deep Learning: Multi-Layer Neural Networks (2–3 Layers)

A single neuron (Perceptron or Sigmoid) can only draw a **single straight line** (a linear decision boundary). Real-world data (images, text, curved patterns) is rarely separable by a straight line.

**Key Idea:** Stack neurons into **layers**!
- **Input Layer:** Raw features ($x_1, x_2, \dots$)
- **Hidden Layer(s):** Combine inputs into intermediate features (e.g., edges -> shapes -> objects)
- **Output Layer:** Produces the final prediction

When we stack multiple hidden layers, we enter **Deep Learning**.

### Layer-by-Layer Matrix Equations
For each layer $l$:
$$Z^{[l]} = A^{[l-1]} W^{[l]} + b^{[l]}$$
$$A^{[l]} = f(Z^{[l]})$$
*(where $A^{[0]} = X$ is the input data)*

In [ ]:
# Helper to draw a Multi-Layer Neural Network Architecture
def draw_neural_net(layer_sizes, layer_names, title):
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.axis('off')
    
    x_positions = np.linspace(1, 9, len(layer_sizes))
    colors = ['#dceefb', '#fff3cd', '#ffe0b2', '#e8f5e9']
    edge_colors = ['#1f77b4', '#ff9800', '#ef6c00', '#2e7d32']
    
    node_coords = []
    for i, n_nodes in enumerate(layer_sizes):
        y_positions = np.linspace(1.2, 5.2, n_nodes) if n_nodes > 1 else [3.2]
        coords = [(x_positions[i], y) for y in y_positions]
        node_coords.append(coords)
        
    # Draw connections
    for l in range(len(node_coords) - 1):
        for (x1, y1) in node_coords[l]:
            for (x2, y2) in node_coords[l + 1]:
                ax.plot([x1, x2], [y1, y2], color='#aaaaaa', lw=1.0, zorder=1)
                
    # Draw nodes
    for l, coords in enumerate(node_coords):
        c_idx = min(l, len(colors) - 1)
        for idx, (x, y) in enumerate(coords):
            circle = Circle((x, y), 0.28, facecolor=colors[c_idx], edgecolor=edge_colors[c_idx], lw=2, zorder=2)
            ax.add_patch(circle)
        ax.text(x_positions[l], 0.4, layer_names[l], ha='center', fontsize=11, fontweight='bold', color=edge_colors[c_idx])
        
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

draw_neural_net(
    layer_sizes=[2, 4, 3, 1],
    layer_names=['Input Layer\n(2 features)', 'Hidden Layer 1\n(4 neurons)', 'Hidden Layer 2\n(3 neurons)', 'Output Layer\n(1 neuron)'],
    title='A 3-Layer Neural Network (2 Hidden Layers + 1 Output Layer)'
)

### Forward Pass Through a 3-Layer Network in Pure NumPy
Let's trace a single data point through the exact `2 -> 4 -> 3 -> 1` network drawn above so we can see the matrix multiplications clearly.

In [ ]:
# 1 sample with 2 features
X_sample = np.array([[0.5, -1.2]])  # shape: (1, 2)

# Random weights and biases for 3 layers: (2 -> 4 -> 3 -> 1)
W1, b1 = np.random.randn(2, 4), np.zeros((1, 4))
W2, b2 = np.random.randn(4, 3), np.zeros((1, 3))
W3, b3 = np.random.randn(3, 1), np.zeros((1, 1))

# Forward Pass
A1 = sigmoid(X_sample @ W1 + b1)   # Hidden Layer 1
A2 = sigmoid(A1 @ W2 + b2)         # Hidden Layer 2
y_hat = sigmoid(A2 @ W3 + b3)      # Output Layer

print(f"Input shape:          {X_sample.shape} -> {X_sample}")
print(f"Hidden Layer 1 shape: {A1.shape} -> {np.round(A1, 3)}")
print(f"Hidden Layer 2 shape: {A2.shape} -> {np.round(A2, 3)}")
print(f"Final Output shape:   {y_hat.shape} -> {y_hat[0, 0]:.4f}")

---
# 5. Building a Classification Neural Network

Now let's **train** a multi-layer neural network using Gradient Descent (via Backpropagation) on a non-linear dataset (`make_moons`) and compare it directly against a single neuron (`LogisticRegression`).

In [ ]:
# Generate non-linear classification data
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)
print(f"Dataset shape: {X_moons.shape}, Classes: {np.unique(y_moons)}")

plt.figure(figsize=(6, 4.5))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap=plt.cm.coolwarm, edgecolors='k')
plt.title('Non-Linear Binary Classification Data (Make Moons)')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.show()

In [ ]:
# Helper function to plot decision boundaries
def plot_decision_boundary(model, X, y, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4.5))
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors='k', s=35)
    ax.set_title(title)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

In [ ]:
# 1. Single Neuron (Logistic Regression)
single_neuron = LogisticRegression()
single_neuron.fit(X_moons, y_moons)

# 2. 3-Layer Neural Network (2 Hidden Layers: 16 neurons, then 8 neurons)
nn_classifier = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', max_iter=1000, random_state=42)
nn_classifier.fit(X_moons, y_moons)

print(f"Single Neuron Accuracy:  {single_neuron.score(X_moons, y_moons):.2%}")
print(f"Neural Network Accuracy: {nn_classifier.score(X_moons, y_moons):.2%}")

In [ ]:
# Compare decision boundaries side-by-side
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
plot_decision_boundary(single_neuron, X_moons, y_moons, 'Single Neuron (Linear Boundary)', ax=axes[0])
plot_decision_boundary(nn_classifier, X_moons, y_moons, '3-Layer Neural Network (Non-Linear Boundary)', ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# Connect back to Gradient Descent: plot how the loss decreased during training
plt.figure(figsize=(7, 3.5))
plt.plot(nn_classifier.loss_curve_, color='#1f77b4', lw=2)
plt.title('Neural Network Training Loss Curve (Gradient Descent in Action)')
plt.xlabel('Iteration (Epoch)')
plt.ylabel('Cross-Entropy Loss')
plt.grid(alpha=0.3)
plt.show()

---
# 6. Multi-Class Classification & Softmax

For **2 classes** (binary), a single output neuron with **Sigmoid** gives $P(\text{Class 1})$, and $P(\text{Class 0}) = 1 - P(\text{Class 1})$.

What if we have **3 or more classes** (e.g., Cat, Dog, Bird)?
- Our output layer now has **$K$ neurons** (one per class), each producing a raw score called a **logit** ($z_1, z_2, \dots, z_K$).
- Logits can be negative or greater than 1, and they don't sum to 1.

### The Softmax Function
Softmax converts raw logits into a valid **probability distribution** where every value is between $0$ and $1$, and all probabilities sum to $1$:

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

- Taking $e^{z_i}$ makes every score **positive** and **amplifies the largest score**.
- Dividing by the sum normalizes them so they **add up to 100% (1.0)**.

In [ ]:
# Implement Softmax from scratch (subtracting max(z) prevents numerical overflow)
def softmax(z):
    exp_z = np.exp(z - np.max(z))
    return exp_z / np.sum(exp_z)

classes = ['Cat', 'Dog', 'Bird']
logits = np.array([2.5, 1.0, -0.5])
probabilities = softmax(logits)

for c, l, p in zip(classes, logits, probabilities):
    print(f"{c:5s} | Raw Logit: {l:5.2f} -> Softmax Probability: {p:.4f} ({p*100:.1f}%)")
print(f"Sum of probabilities: {np.sum(probabilities):.4f}")

In [ ]:
# Visualize Raw Logits vs. Softmax Probabilities
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].bar(classes, logits, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='k')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Raw Output Scores (Logits)')
axes[0].set_ylabel('Score (z)')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(classes, probabilities, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='k')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('After Softmax (Probabilities Sum to 1.0)')
axes[1].set_ylabel('Probability')
axes[1].grid(axis='y', alpha=0.3)
for i, p in enumerate(probabilities):
    axes[1].text(i, p + 0.03, f'{p*100:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Multi-Class Neural Network in Action (3 Classes)
Let's generate a 3-class dataset, train a neural network (which automatically uses **Softmax** at the output layer for multi-class problems), inspect the predicted probabilities, and plot the 3 decision regions.

In [ ]:
# Generate 3-class data
X_multi, y_multi = make_blobs(n_samples=300, centers=3, cluster_std=1.5, random_state=42)

plt.figure(figsize=(6, 4.5))
plt.scatter(X_multi[:, 0], X_multi[:, 1], c=y_multi, cmap=plt.cm.viridis, edgecolors='k')
plt.title('3-Class Dataset')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.show()

In [ ]:
# Train a 3-class Neural Network (uses Softmax at the 3-neuron output layer)
multi_nn = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', max_iter=1000, random_state=42)
multi_nn.fit(X_multi, y_multi)

# Inspect Softmax probabilities for the first 5 points
sample_probs = multi_nn.predict_proba(X_multi[:5])
print("Predicted Softmax Probabilities for 5 samples [Class 0, Class 1, Class 2]:")
for i, p in enumerate(sample_probs):
    print(f"Sample {i}: {np.round(p, 3)} | Sum = {p.sum():.2f} | Predicted Class = {np.argmax(p)}")

In [ ]:
# Plot the 3-Class Decision Regions
h = 0.05
x_min, x_max = X_multi[:, 0].min() - 1, X_multi[:, 0].max() + 1
y_min, y_max = X_multi[:, 1].min() - 1, X_multi[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z = multi_nn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6.5, 5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.viridis)
plt.scatter(X_multi[:, 0], X_multi[:, 1], c=y_multi, cmap=plt.cm.viridis, edgecolors='k', s=40)
plt.title('3-Class Neural Network Decision Boundaries (Softmax Output)')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.show()